### **SPARK STREAMING**

In [0]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, DateType, TimestampType
)
import logging

In [0]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

In [0]:
entities = ["customers","drivers","locations","payments","trips","vehicles"]

In [0]:
customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("city", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("last_updated_timestamp", TimestampType(), True),
])

driver_schema = StructType([
    StructField("driver_id", IntegerType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("phone_number", StringType(), True),
    StructField("vehicle_id", IntegerType(), True),
    StructField("driver_rating", DoubleType(), True),
    StructField("city", StringType(), True),
    StructField("last_updated_timestamp", TimestampType(), True),
])
location_schema = StructType([
    StructField("location_id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("last_updated_timestamp", TimestampType(), True),
])
vehicle_schema = StructType([
    StructField("vehicle_id", IntegerType(), True),
    StructField("license_plate", StringType(), True),
    StructField("model", StringType(), True),
    StructField("make", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("vehicle_type", StringType(), True),
    StructField("last_updated_timestamp", TimestampType(), True),
])
trip_schema = StructType([
    StructField("trip_id", IntegerType(), True),
    StructField("driver_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("vehicle_id", IntegerType(), True),
    StructField("trip_start_time", TimestampType(), True),
    StructField("trip_end_time", TimestampType(), True),
    StructField("start_location", StringType(), True),
    StructField("end_location", StringType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("trip_status", StringType(), True),
    StructField("last_updated_timestamp", TimestampType(), True),
])
payment_schema = StructType([
    StructField("payment_id", IntegerType(), True),
    StructField("trip_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("payment_method", StringType(), True),
    StructField("payment_status", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("transaction_time", TimestampType(), True),
    StructField("last_updated_timestamp", TimestampType(), True),
])

In [0]:
schemas = {
    "customers": customers_schema,
    "drivers": driver_schema,
    "locations": location_schema,
    "vehicles": vehicle_schema,
    "trips": trip_schema,
    "payments": payment_schema,
}

In [0]:
for entity in entities:
    try:
        logger.info(f"Bronze ingestion in progress: {entity}")
        schema_entity = schemas[entity]

        df_stream = spark.readStream.format("csv")\
        .option("header", True)\
        .schema(schema_entity)\
        .load(f"/Volumes/pysparkdbt/source/source_data/{entity}")

        df_stream.writeStream.format("delta")\
        .outputMode("append")\
        .option("checkpointLocation",f"/Volumes/pysparkdbt/bronze/checkpoint/{entity}")\
        .trigger(once=True)\
        .toTable(f"pysparkdbt.bronze.{entity}")

        logger.info(f"Bronze ingestion completed: {entity}")
    except Exception as e:
        logger.error(f"Bronze ingestion failed for {entity}: {e}")
        raise